In [ ]:
import json, re, time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
MODEL_ID = "Qwen/Qwen3-8B"
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
)
model.eval()
 
with open("Examples.json") as f:
    examples = json.load(f)
with open("Task.json") as f:
    tasks = json.load(f)
 
feast_by_level = {}
block_by_level = {}
for e in examples:
    lvl = e["complexity_level"]
    if "feast" in e["scenario_context"].lower() or "attack object" in e["scenario_context"].lower():
        if lvl not in feast_by_level:
            feast_by_level[lvl] = e
    if "block" in e["scenario_context"].lower():
        if lvl not in block_by_level:
            block_by_level[lvl] = e
 
def build_prompt(scenario_context):
    is_block = "block" in scenario_context.lower()
    pool = block_by_level if is_block else feast_by_level
    prefix = ""
    for lvl in sorted(pool.keys()):
        e = pool[lvl]
        ctx = e["scenario_context"]
        plan_end = ctx.rfind("[PLAN END]")
        if plan_end != -1:
            prefix += ctx[:plan_end + len("[PLAN END]")] + "\n\n"
    return prefix + scenario_context
 
def parse_plan(generated_text, scenario_context):
    is_block = "block" in scenario_context.lower()
    after = generated_text.split("[PLAN]")[-1]
    end_idx = after.find("[PLAN END]")
    plan_text = after[:end_idx].strip() if end_idx != -1 else after.strip()
    lines = [l.strip().rstrip(".") for l in plan_text.splitlines() if l.strip()]
 
    actions = []
    for line in lines:
        if is_block:
            m = re.match(r"(unmount_node|mount_node)\s+(?:the\s+)?(\w+)(?:\s+block)?\s+(?:from\s+on\s+top\s+of|on\s+top\s+of)\s+(?:the\s+)?(\w+)", line, re.I)
            if m:
                actions.append(f"({m.group(1).lower()} {m.group(2).lower()} {m.group(3).lower()})")
                continue
            m = re.match(r"(engage_payload|release_payload)\s+(?:the\s+)?(\w+)", line, re.I)
            if m:
                actions.append(f"({m.group(1).lower()} {m.group(2).lower()})")
                continue
            m = re.match(r"pick\s+up\s+(?:the\s+)?(\w+)", line, re.I)
            if m:
                actions.append(f"(engage_payload {m.group(1).lower()})")
                continue
            m = re.match(r"put\s+down\s+(?:the\s+)?(\w+)", line, re.I)
            if m:
                actions.append(f"(release_payload {m.group(1).lower()})")
                continue
            m = re.match(r"(?:stack|mount_node)\s+(?:the\s+)?(\w+)(?:\s+block)?\s+on\s+top\s+of\s+(?:the\s+)?(\w+)", line, re.I)
            if m:
                actions.append(f"(mount_node {m.group(1).lower()} {m.group(2).lower()})")
                continue
            m = re.match(r"(?:unstack|unmount_node)\s+(?:the\s+)?(\w+)(?:\s+block)?\s+from\s+(?:on\s+top\s+of\s+)?(?:the\s+)?(\w+)", line, re.I)
            if m:
                actions.append(f"(unmount_node {m.group(1).lower()} {m.group(2).lower()})")
        else:
            m = re.match(r"(attack|succumb|feast|overcome)\s+object\s+(\w+)(?:\s+from\s+object\s+(\w+))?", line, re.I)
            if m:
                action, obj1 = m.group(1).lower(), m.group(2).lower()
                obj2 = m.group(3).lower() if m.group(3) else None
                actions.append(f"({action} {obj1} {obj2})" if obj2 else f"({action} {obj1})")
    return actions
 
def infer_complexity(n):
    if n <= 2: return 2
    if n <= 4: return 4
    return 6
 
start = time.time()
results = []
 
for i, task in enumerate(tasks):
    ctx = task["scenario_context"]
    prompt = build_prompt(ctx)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
    input_len = inputs["input_ids"].shape[1]
 
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
 
    generated = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    actions = parse_plan(prompt + generated, ctx)
    complexity = infer_complexity(len(actions))
 
    results.append({
        "assembly_task_id": task["assembly_task_id"],
        "complexity_level": complexity,
        "target_action_sequence": actions,
    })
 
    elapsed = time.time() - start
    print(f"[{i+1}/{len(tasks)}] {task['assembly_task_id']} | complexity={complexity} | actions={len(actions)} | t={elapsed:.1f}s")
 
print(f"\nTotal: {time.time()-start:.1f}s")
 
with open("output.json", "w") as f:
    json.dump(results, f, indent=2)
 
print("Guardado en output.json")
 